In [10]:
# Instalações (para Google Colab)
!apt-get update > /dev/null
!apt install chromium-chromedriver -y > /dev/null
!cp /usr/lib/chromium-browser/chromedriver /usr/bin
!pip install selenium pandas openpyxl > /dev/null

# Imports
import os, re, time, requests, zipfile
from datetime import datetime
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

# Setup de pastas com data
data_hoje = datetime.now().strftime('%Y-%m-%d')
pasta_base = f"{data_hoje}_casa_rossi"
pasta_imagens = os.path.join(pasta_base, "imagens")
os.makedirs(pasta_imagens, exist_ok=True)

# Configurar WebDriver (headless)
options = Options()
options.binary_location = "/usr/bin/chromium-browser"
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
driver = webdriver.Chrome(options=options)

# Coletar links dos produtos
url_lista = 'https://www.casarossi.com.br/pisos-e-revestimentos/porcelanatos/acetinado'
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url_lista, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

links_produtos = []
produtos = soup.find_all('li', class_='item flex')
for produto in produtos:
    a_tag = produto.find('a', class_='space-image')
    if a_tag:
        link = a_tag['href']
        if not link.startswith("http"):
            link = "https://www.casarossi.com.br" + link
        links_produtos.append(link)

print(f"🔗 {len(links_produtos)} produtos encontrados.")

# Função para salvar imagem
def salvar_imagem(url_img, nome_base):
    if url_img == "N/A":
        return
    try:
        resposta = requests.get(url_img, timeout=10)
        if resposta.status_code == 200:
            caminho = os.path.join(pasta_imagens, f"{nome_base}.jpg")
            with open(caminho, 'wb') as f:
                f.write(resposta.content)
    except Exception as e:
        print(f"❌ Erro ao baixar imagem: {e}")

# Extrair dados dos produtos
dados_produtos = []
for i, url in enumerate(links_produtos):
    print(f"[{i+1}/{len(links_produtos)}] Coletando dados de: {url}")
    try:
        driver.get(url)
        time.sleep(3)
        soup = BeautifulSoup(driver.page_source, 'html.parser')

        nome = limpar(soup.find('h1', class_='product-name').text)
        preco_caixa = soup.find("span", class_="PrecoPrincipal").get_text(strip=True).replace("R$", "") if soup.find("span", class_="PrecoPrincipal") else "N/A"
        preco_m2 = soup.find("div", class_="preco-por-metro").get_text(strip=True).replace("R$", "") if soup.find("div", class_="preco-por-metro") else "N/A"
        m2_caixa = soup.find("div", class_="metro-na-caixa").get_text(strip=True) if soup.find("div", class_="metro-na-caixa") else "N/A"
        descricao = next((p.get_text(separator='\n', strip=True) for p in soup.find_all('p') if 'Informações do Produto' in p.get_text()), "N/A")

        try:
            img_tag = soup.find("div", class_="zoom").find("img")
            imagem_url = img_tag["src"]
        except:
            imagem_url = "N/A"

        # Criar código com base no nome
        codigo_base = re.sub(r'[\\/*?:"<>|]', "", nome)[:50]
        salvar_imagem(imagem_url, codigo_base)

        dados_produtos.append([
            codigo_base,
            nome,
            preco_caixa,
            "",  # Preço Original não disponível
            preco_m2,
            f"m² por caixa: {m2_caixa}",
            descricao,
            imagem_url,
            url,
            "Casa Rossi"
        ])

    except Exception as e:
        print(f"❌ Erro ao processar {url}: {e}")

driver.quit()

# Criar DataFrame
df = pd.DataFrame(dados_produtos, columns=[
    "Código", "Nome", "Preço Promocional", "Preço Original", "Preço por m²",
    "Especificações", "Descrição", "Imagem", "URL", "Loja"
])

# Função para limpar preços
def limpar_preco(valor):
    try:
        return float(valor.replace("R$", "").replace(".", "").replace(",", ".").strip())
    except:
        return None

df["Preço Promocional"] = df["Preço Promocional"].apply(limpar_preco)
df["Preço Original"] = df["Preço Original"].apply(limpar_preco)
df["Preço por m²"] = df["Preço por m²"].apply(limpar_preco)

# Salvar CSV
csv_path = os.path.join(pasta_base, f"casa_rossi_{data_hoje}.csv")
df.to_csv(csv_path, sep=';', index=False, encoding='utf-8-sig')
print(f"📄 CSV salvo em: {csv_path}")

# Salvar Excel
xlsx_path = os.path.join(pasta_base, f"casa_rossi_{data_hoje}.xlsx")
df.to_excel(xlsx_path, index=False)
print(f"📊 Excel salvo em: {xlsx_path}")

# Compactar imagens
zip_path = os.path.join(pasta_base, f"imagens_casa_rossi_{data_hoje}.zip")
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for root, dirs, files in os.walk(pasta_imagens):
        for file in files:
            zipf.write(os.path.join(root, file), arcname=file)

print(f"📦 Imagens compactadas em: {zip_path}")


FileNotFoundError: [Errno 2] No such file or directory: 'produto_exemplo.html'

In [11]:
# Instalações (apenas no Google Colab)
!apt-get update > /dev/null
!apt install chromium-chromedriver -y > /dev/null
!cp /usr/lib/chromium-browser/chromedriver /usr/bin
!pip install selenium pandas openpyxl > /dev/null

# Imports
import os, re, time, requests, zipfile
from datetime import datetime
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

# Função para limpar strings
def limpar(texto):
    return ' '.join(texto.strip().split()) if texto else ''

# Setup de pastas
data_hoje = datetime.now().strftime('%Y-%m-%d')
pasta_base = f"{data_hoje}_casa_rossi"
pasta_imagens = os.path.join(pasta_base, "imagens")
os.makedirs(pasta_imagens, exist_ok=True)

# Configurar WebDriver (headless)
options = Options()
options.binary_location = "/usr/bin/chromium-browser"
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
driver = webdriver.Chrome(options=options)

# Coletar links dos produtos
url_lista = 'https://www.casarossi.com.br/pisos-e-revestimentos/porcelanatos/acetinado'
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url_lista, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

links_produtos = []
for produto in soup.find_all('li', class_='item flex'):
    a_tag = produto.find('a', class_='space-image')
    if a_tag:
        link = a_tag['href']
        if not link.startswith("http"):
            link = "https://www.casarossi.com.br" + link
        links_produtos.append(link)

print(f"🔗 {len(links_produtos)} produtos encontrados.")

# Função para salvar imagem
def salvar_imagem(url_img, nome_arquivo):
    if not url_img:
        return
    try:
        resposta = requests.get(url_img, timeout=10)
        if resposta.status_code == 200:
            caminho = os.path.join(pasta_imagens, nome_arquivo)
            with open(caminho, 'wb') as f:
                f.write(resposta.content)
    except Exception as e:
        print(f"❌ Erro ao baixar imagem: {e}")

# Função para extrair dados de um produto
def extrair_dados_produto(html, url):
    soup = BeautifulSoup(html, 'html.parser')

    nome = limpar(soup.find('h1', class_='product-name').text)
    codigo = limpar(soup.select_one('.product-sku .ref').text) if soup.select_one('.product-sku .ref') else re.sub(r'[\\/*?:"<>|]', "", nome)[:50]
    marca = limpar(soup.find_all('div', class_='product-brand')[0].text.replace('Marca:', '')) if soup.find_all('div', class_='product-brand') else ''
    modelo = limpar(soup.find_all('div', class_='product-brand')[1].text.replace('Modelo:', '')) if len(soup.find_all('div', class_='product-brand')) > 1 else ''

    preco_m2 = limpar(soup.select_one('.preco-por-metro').text.replace('por m²', '').replace('R$', '')) if soup.select_one('.preco-por-metro') else ''
    m2_caixa = limpar(soup.select_one('.metro-na-caixa').text.replace('Caixa com', '').replace('m²', '')) if soup.select_one('.metro-na-caixa') else ''
    preco_caixa = limpar(soup.select_one('.PrecoPrincipal span').text.replace('R$', '')) if soup.select_one('.PrecoPrincipal span') else ''

    imagem_tag = soup.select_one('.zoom img')
    imagem_url = imagem_tag['src'] if imagem_tag else ''
    imagem_ext = imagem_url.split('.')[-1].split('?')[0] if imagem_url else 'jpg'
    imagem_nome = f"{codigo}.{imagem_ext}"

    descricao_html = soup.find('p')
    descricao = limpar(descricao_html.get_text(separator=' ')) if descricao_html else ''

    salvar_imagem(imagem_url, imagem_nome)

    return {
        'Código': codigo,
        'Nome': nome,
        'Marca': marca,
        'Modelo': modelo,
        'Preço por m²': preco_m2,
        'Preço por Caixa': preco_caixa,
        'm² por Caixa': m2_caixa,
        'Imagem': imagem_nome,
        'Descrição': descricao,
        'URL': url,
        'Loja': 'Casa Rossi'
    }

# Coleta dos dados
dados_produtos = []
for i, url in enumerate(links_produtos):
    print(f"[{i+1}/{len(links_produtos)}] {url}")
    try:
        driver.get(url)
        time.sleep(3)
        html = driver.page_source
        dados = extrair_dados_produto(html, url)
        dados_produtos.append(dados)
    except Exception as e:
        print(f"❌ Erro ao processar {url}: {e}")

driver.quit()

# Criar DataFrame
df = pd.DataFrame(dados_produtos)

# Padronizar preços
def limpar_preco(valor):
    try:
        return float(valor.replace('.', '').replace(',', '.'))
    except:
        return None

df["Preço por m²"] = df["Preço por m²"].apply(limpar_preco)
df["Preço por Caixa"] = df["Preço por Caixa"].apply(limpar_preco)

# Salvar CSV
csv_path = os.path.join(pasta_base, f"casa_rossi_{data_hoje}.csv")
df.to_csv(csv_path, sep=';', index=False, encoding='utf-8-sig')
print(f"📄 CSV salvo em: {csv_path}")

# Salvar Excel
xlsx_path = os.path.join(pasta_base, f"casa_rossi_{data_hoje}.xlsx")
df.to_excel(xlsx_path, index=False)
print(f"📊 Excel salvo em: {xlsx_path}")

# Compactar imagens
zip_path = os.path.join(pasta_base, f"imagens_casa_rossi_{data_hoje}.zip")
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for root, _, files in os.walk(pasta_imagens):
        for file in files:
            zipf.write(os.path.join(root, file), arcname=file)

print(f"📦 Imagens compactadas em: {zip_path}")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


cp: '/usr/lib/chromium-browser/chromedriver' and '/usr/bin/chromedriver' are the same file
🔗 15 produtos encontrados.
[1/15] https://www.casarossi.com.br/pisos-e-revestimentos/porcelanatos/acetinado/piso-formigres-66x66cm-acetinado-retificado-coimbra-2-18m
[2/15] https://www.casarossi.com.br/pisos-e-revestimentos/porcelanatos/acetinado/porcelanato-retificado-delta-72x72cm-out-itaara-raven-2-61m
[3/15] https://www.casarossi.com.br/pisos-e-revestimentos/porcelanatos/acetinado/porcelanatoincepa-90x90cm-acetinado-duo-concret-cris-2-40m
[4/15] https://www.casarossi.com.br/pisos-e-revestimentos/porcelanatos/acetinado/porcelanato-helena-62x121cm-acetinado-120133-mint-2-25m
[5/15] https://www.casarossi.com.br/pisos-e-revestimentos/porcelanatos/acetinado/porcelanato-retificado-biancogres-90x90cm-acetinado-c